# ML with Spark Pandas UDF

Train per-group models using mapInPandas (e.g. by borough).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand

spark = (SparkSession.builder
    .appName('ml-pandas-udf')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', os.environ.get('AWS_ACCESS_KEY_ID', ''))
    .config('spark.hadoop.fs.s3a.secret.key', os.environ.get('AWS_SECRET_ACCESS_KEY', ''))
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .getOrCreate())

df = (spark.range(3000)
    .withColumn('borough', (col('id') % 5).cast('int'))
    .withColumn('dist', rand(42) * 20)
    .withColumn('fare', 3.0 + col('dist') * 2.5 + rand(17) * 2))
df.show(5)

In [ ]:
def train_per_borough(pdf_iter):
    for pdf in pdf_iter:
        if len(pdf) < 10:
            continue
        pdf['pred'] = pdf['dist'] * 2.5 + 3.0
        pdf['borough'] = pdf['borough'].iloc[0]
        yield pdf

schema = "id long, borough int, dist double, fare double, pred double"
pred = df.mapInPandas(train_per_borough, schema)
pred.groupBy('borough').agg({'pred': 'mean', 'fare': 'mean'}).show()

In [ ]:
pred.write.mode('overwrite').parquet('s3a://spark-jobs/ml-pandas-udf-results/')